In [ ]:
import os
import warnings
import sleap
import numpy as np

# Print some info:
sleap.versions()
sleap.system_summary()

# Load files

In [ ]:
labels_ch = sleap.load_file("long_id.slp")
labels_ap = sleap.load_file("long_full_pose.slp")
labels_ap_full = labels_ap.copy()
if os.path.basename(os.path.normpath(labels_ch.video.filename)) != os.path.basename(os.path.normpath(labels_ap.video.filename)):
    raise ValueError("Video files don't match!")

In [ ]:
if labels_ch.video.filename != labels_ap.video.filename:
    print("Video file paths don't match, but filenames do. Making paths match...")
    videos_ap = labels_ap.video
    videos_ch = labels_ch.video
    videos_ap.backend.filename = videos_ch.backend.filename
    print("Done")

In [ ]:
nodes_to_remove = ['nose', 'head', 'right_ear', 'left_ear', 'spine1', 'spine3', 'spine4']
for node in nodes_to_remove:
    labels_ap.skeleton.delete_node(node)

# Match instances and create full pose .slp file with ID

In [ ]:
# Match frames and instances
frame_pairs = sleap.nn.evals.find_frame_pairs(labels_ch, labels_ap, user_labels_only=False)
matched_instances = sleap.nn.evals.match_frame_pairs(frame_pairs, stddev = 0.05, scale = 2.5, threshold=0, user_labels_only=False)

j = 0
instances = []
assigned_tracks = []
assigned_instances = []
frame_idx = matched_instances[0][0][0].frame_idx
full_pose_frame_data = labels_ap_full.find(labels_ap_full.video, frame_idx)[0]
candidate_points = [instance.points_array[5] for instance in full_pose_frame_data.instances]
lfs=[]
# Loop over instances
for j in range(len(matched_instances[0])):
    # Find the index of the mouse who was matched to the current identity we are considering (matched_instance[0])
    # This is done in order to get back the full pose (labels_ap only has spine2)
    instance_idx = np.where(candidate_points == matched_instances[0][j][1].points_array[0])[0][0]
    instances.append(
        sleap.PredictedInstance(
            skeleton=labels_ap_full.skeleton,
            track=matched_instances[0][j][0].track,
            points=full_pose_frame_data[instance_idx].get_points_array(copy=False, full=True),
        )
    )
    assigned_tracks.append(matched_instances[0][j][0].track)
    assigned_instances.append(instance_idx)
    # Update the frame if we are at the end of the current frame
    if j < len(matched_instances[0])-1 and matched_instances[0][j+1][0].frame_idx != matched_instances[0][j][0].frame_idx:
        # If only one track from the current frame has not been assigned and labels_ap has detected all instances, 
        # add the unmatched instance and assign it the missing track
        if len(assigned_tracks) == len(labels_ch.tracks)-1 and len(full_pose_frame_data.instances) == len(labels_ch.tracks):
            missing_track = list(set(labels_ch.tracks) - set(assigned_tracks))[0]
            missing_instance = list(set([0, 1, 2]) - set(assigned_instances))[0]
            instances.append(
                sleap.PredictedInstance(
                    skeleton=labels_ap_full.skeleton,
                    track=missing_track,
                    points=full_pose_frame_data[missing_instance].get_points_array(copy=False, full=True),
                )
            )
        lf = sleap.instance.LabeledFrame(
            video=labels_ap.video,
            frame_idx=frame_idx,
            instances=instances,
        )
        lfs.append(lf)
        instances = []
        assigned_tracks = []
        assigned_instances = []
        frame_idx = matched_instances[0][j+1][0].frame_idx
        full_pose_frame_data = labels_ap_full.find(labels_ap_full.video, frame_idx)[0]
        candidate_points = [instance.points_array[5] for instance in full_pose_frame_data.instances]
labels_ap_id = sleap.Labels(labeled_frames=lfs)
sleap.Labels.save_file(labels_ap_id, "full_pose_w_id.slp")